# Bank Marketing — Exploratory Data Analysis

**Purpose**: Understand the raw dataset before modelling. Covers structural inspection, nulls,
class imbalance (target), feature distributions, and correlation analysis.

**Entrypoint**: All paths and parameters are driven by `../config.yaml`.
No values are hardcoded in this notebook.

## 1. Import Required Libraries

In [ ]:
import sys
import os
import warnings

# Allow importing src/ from the notebooks/ directory
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress known harmless warnings; leave data warnings visible.
# Pandas4Warning: select_dtypes("object") deprecation for pandas 3 — safe to ignore on pandas 2.2.x.
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

from src.data import load_data
from src.config import load_config

CONFIG_PATH = "../config.yaml"

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_theme(style="whitegrid")

## 2. Load Configuration

In [ ]:
config = load_config(CONFIG_PATH)

TARGET = config["model"]["target_column"]

print("Config loaded:")
print(f"  Raw data path : {config['data']['raw_path']}")
print(f"  Target column : {TARGET}")
print(f"  Test size     : {config['model']['test_size']}")
print(f"  Random state  : {config['model']['random_state']}")

## 3. Load Raw CSV Data

In [ ]:
df = load_data(config, os.path.dirname(CONFIG_PATH))

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns\n")
df.head()

## 4. Data Inspection & Validation

In [ ]:
# --- Dtypes overview ---
print("=== Column Types ===")
print(df.dtypes.to_string())

print("\n=== Descriptive Statistics ===")
df.describe(include="all")

In [ ]:
# --- Null analysis ---
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct})
null_summary = null_summary[null_summary["null_count"] > 0].sort_values("null_pct", ascending=False)

if null_summary.empty:
    print("No missing values found.")
else:
    print(f"Columns with missing values (flagged if >5%):\n")
    null_summary["flagged"] = null_summary["null_pct"] > 5
    print(null_summary.to_string())

In [ ]:
# --- Class imbalance ---
class_counts = df[TARGET].value_counts()
class_pct = (class_counts / len(df) * 100).round(2)

print(f"=== Target: '{TARGET}' ===")
for label, count in class_counts.items():
    print(f"  {label}: {count:,}  ({class_pct[label]}%)")

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"\nImbalance ratio (majority/minority): {imbalance_ratio:.1f}x")
if imbalance_ratio > 3:
    print("  ⚠ Significant imbalance detected — consider SMOTE or class_weight='balanced' during training.")

fig, ax = plt.subplots(figsize=(6, 4))
class_counts.plot(kind="bar", ax=ax, color=["#2196F3", "#FF7043"], edgecolor="black")
ax.set_title(f"Class Distribution — '{TARGET}'")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_xticklabels(class_counts.index, rotation=0)
for i, (count, pct) in enumerate(zip(class_counts, class_pct)):
    ax.text(i, count + 20, f"{pct}%", ha="center", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Feature Distributions

In [ ]:
# --- Numeric feature distributions ---
numeric_cols = df.select_dtypes(include=np.number).columns.drop(TARGET, errors="ignore")

if len(numeric_cols) == 0:
    print("No numeric features found.")
else:
    n_cols = 3
    n_rows = int(np.ceil(len(numeric_cols) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 3.5), squeeze=False)
    axes = axes.flatten()

    for i, col in enumerate(numeric_cols):
        axes[i].hist(df[col].dropna(), bins=40, color="#2196F3", edgecolor="white", alpha=0.8)
        axes[i].set_title(col)
        axes[i].set_ylabel("Count")

    # Hide unused axes
    for j in range(len(numeric_cols), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Numeric Feature Distributions", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Categorical feature distributions ---
cat_cols = df.select_dtypes(include="object").columns.drop(TARGET, errors="ignore")

if len(cat_cols) == 0:
    print("No categorical features found.")
else:
    n_cols = 2
    n_rows = int(np.ceil(len(cat_cols) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 4), squeeze=False)
    axes = axes.flatten()

    for i, col in enumerate(cat_cols):
        counts = df[col].value_counts()
        axes[i].bar(counts.index, counts.values, color="#FF7043", edgecolor="white", alpha=0.85)
        axes[i].set_title(col)
        axes[i].set_ylabel("Count")
        axes[i].tick_params(axis="x", rotation=30)

    # Hide unused axes
    for j in range(len(cat_cols), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle("Categorical Feature Distributions", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

## 6. Correlation Analysis

In [ ]:
# Pearson correlation on numeric columns only (excludes categorical)
CORR_THRESHOLD = 0.8

corr_cols = df.select_dtypes(include=np.number).columns
corr_matrix = df[corr_cols].corr()

# Identify strongly correlated pairs (|r| > threshold, upper triangle only)
strong_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i + 1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > CORR_THRESHOLD:
            strong_pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r, 3)))

print(f"Pairs with |r| > {CORR_THRESHOLD}:")
if strong_pairs:
    for col_a, col_b, r in sorted(strong_pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f"  {col_a}  ↔  {col_b}  (r = {r})")
else:
    print("  None found — no multicollinearity concerns.")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # Show lower triangle only
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 8},
)
ax.set_title("Pearson Correlation Matrix (Numeric Features)", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Key Observations

| Area | Observation |
|---|---|
| Shape | 4,566 rows × 18 columns (17 features + target) |
| Missing values | `job` 1.95% (89), `balance` 2.96% (135), `contact` 2.91% (133), `education` 3.00% (137) — all below 5% threshold, no column flagged for drop |
| Class imbalance | `no`: 88.4% (4,038), `yes`: 11.6% (528) — ratio **7.6:1**. Significant imbalance. Use `class_weight='balanced'` during training; evaluate with F1/AUC-ROC, not accuracy |
| Suspicious values | `age` max = 131 (4 rows > 100) — likely data entry errors. Cap or remove during preprocessing |
| Skewed distributions | `duration` (skew=2.76), `balance` (skew=6.55), `campaign` (skew=4.73) — all right-skewed with outliers. Apply log/sqrt transform in `src/features.py` |
| `pdays` encoding | Value `-1` means "not previously contacted" — acts as a sentinel, not a true numeric. Consider encoding as a binary flag + separate numeric |
| Correlated features | `pdays` ↔ `previous` (r=0.577) — moderate correlation; no perfect multicollinearity (all pairs below \|r\|=0.8); safe to keep both |
| Recommended actions | 1. Drop or cap 4 rows with `age > 100` · 2. Impute nulls (median for `balance`, mode for `job`/`education`/`contact`) · 3. Log-transform `duration`, `balance`, `campaign` · 4. Encode `pdays=-1` as `contacted_before` binary flag · 5. Use `class_weight='balanced'` in model training |